# Free GPU Coding-LLM Server on Kaggle (2x T4)

Hosts a coding model on Kaggle GPUs using vLLM and exposes an OpenAI-compatible API through a Cloudflare quick tunnel.

## Before you run
1. In Notebook Settings, choose **GPU T4 x2** and enable Internet.
2. Run cells from top to bottom. The tunnel URL is printed by the **TUNNEL** cell.
3. Kaggle sessions are time-limited; when the session ends, restart the notebook and obtain a new URL.

## Notes
- GPU model availability and memory vary. The configuration cell selects a model based on detected GPU capability and total VRAM.
- Model throughput depends on model, context length, and session load; treat speed estimates as approximate.
- Keep the API key private. A public quick tunnel makes the endpoint reachable from the internet.


In [ ]:
# ============ CONFIG — auto-picks a model for your GPU ============
import torch

FORCE_MODEL = None  # Set a Hugging Face repo ID to override auto-selection.

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Enable a GPU accelerator in Kaggle settings.')

NUM_GPUS = torch.cuda.device_count()
cap = torch.cuda.get_device_capability(0)
SM = cap[0] * 10 + cap[1]
VRAM_TOTAL = sum(torch.cuda.get_device_properties(i).total_memory for i in range(NUM_GPUS)) / 1e9
GPU_NAME = torch.cuda.get_device_name(0)
TP_SIZE = NUM_GPUS if NUM_GPUS in (1, 2, 4) else 1

if FORCE_MODEL:
    MODEL_ID = FORCE_MODEL
elif SM >= 80:
    if VRAM_TOTAL >= 22:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-14B-Instruct'
    elif VRAM_TOTAL >= 14:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    else:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
elif SM >= 75:
    if VRAM_TOTAL >= 24:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-14B-Instruct'
    elif VRAM_TOTAL >= 12:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    else:
        MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
else:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct' if VRAM_TOTAL >= 15 else 'Qwen/Qwen2.5-Coder-3B-Instruct'

MAX_MODEL_LEN = 16384 if VRAM_TOTAL >= 24 else 8192
SERVED_NAME = 'coder'

print(f'GPU: {NUM_GPUS}x {GPU_NAME} (SM{SM}, {VRAM_TOTAL:.0f}GB total)')
print(f'Model: {MODEL_ID} | TP={TP_SIZE} | ctx={MAX_MODEL_LEN}')


In [ ]:
# ============ INSTALL ============
%pip install -q -U vllm huggingface_hub

# Download Cloudflare's quick-tunnel client.
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
print('Install step complete')


In [ ]:
# ============ START vLLM SERVER ============
import os
import subprocess
import time
import urllib.request

API_KEY = os.environ.get('VLLM_API_KEY', 'kaggle-local-key')
cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--served-model-name', SERVED_NAME,
    '--tensor-parallel-size', str(TP_SIZE),
    '--dtype', 'half',
    '--max-model-len', str(MAX_MODEL_LEN),
    '--gpu-memory-utilization', '0.90',
    '--host', '0.0.0.0', '--port', '8000',
    '--api-key', API_KEY,
]
LOG_PATH = '/kaggle/working/vllm.log' if os.path.isdir('/kaggle/working') else 'vllm.log'
log = open(LOG_PATH, 'w')
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print('vLLM is starting; first model download/load may take several minutes...')

for _ in range(240):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2):
            print('vLLM is UP')
            break
    except Exception:
        if server.poll() is not None:
            print('SERVER DIED — last 40 log lines:')
            os.system(f'tail -40 {LOG_PATH}')
            raise RuntimeError('vLLM failed to start. Check the log, reduce MAX_MODEL_LEN, or choose a smaller model.')
        time.sleep(5)
else:
    raise TimeoutError(f'Timed out waiting for vLLM. Check {LOG_PATH}')


In [ ]:
# ============ TUNNEL — prints your public URL ============
import re
import subprocess
import time

tun = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
start = time.time()
while time.time() - start < 60:
    line = tun.stdout.readline()
    if not line:
        time.sleep(0.2)
        if tun.poll() is not None:
            break
        continue
    match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', line)
    if match:
        public_url = match.group(0)
        break

print('=' * 60)
if public_url:
    print('OPENAI-COMPATIBLE BASE URL:', public_url + '/v1')
else:
    print('Public URL not detected. Check Cloudflare output and rerun this cell.')
print('API KEY:', API_KEY)
print('MODEL NAME:', SERVED_NAME)
print('=' * 60)


In [ ]:
# ============ SMOKE TEST + SPEED CHECK ============
import json
import time
import urllib.request

request = urllib.request.Request(
    'http://127.0.0.1:8000/v1/chat/completions',
    data=json.dumps({
        'model': SERVED_NAME,
        'messages': [{'role': 'user', 'content': 'Write a Python function that checks if a string is a palindrome.'}],
        'max_tokens': 256,
    }).encode('utf-8'),
    headers={
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {API_KEY}',
    },
)
t0 = time.time()
with urllib.request.urlopen(request, timeout=300) as response:
    result = json.load(response)
elapsed = max(time.time() - t0, 1e-9)
output_tokens = result.get('usage', {}).get('completion_tokens', 0)
print(result['choices'][0]['message']['content'][:500])
print(f'\\n--- {output_tokens} tokens in {elapsed:.1f}s = {output_tokens / elapsed:.1f} tok/s (single stream) ---')


In [ ]:
# ============ KEEP ALIVE — leave this cell running if desired ============
import time
import urllib.request

while True:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5):
            print(time.strftime('%H:%M:%S'), 'alive')
    except Exception as exc:
        print(time.strftime('%H:%M:%S'), 'server not responding:', exc)
    time.sleep(300)
